# Session 07: Morphological Operations & Contour Analysis

**CVI4IC - Summer Semester 2026**

Cleaning up binary images and extracting shape information.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. Morphological Operations

In [ ]:
# Create a noisy binary image
img = np.zeros((200, 200), dtype=np.uint8)
cv2.rectangle(img, (40, 40), (160, 160), 255, -1)
cv2.circle(img, (100, 100), 30, 0, -1)  # Hole inside

# Add noise
noise = np.random.random(img.shape)
img[noise < 0.02] = 255  # Salt
img[noise > 0.98] = 0    # Pepper

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

eroded = cv2.erode(img, kernel, iterations=1)
dilated = cv2.dilate(img, kernel, iterations=1)
opened = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, image, title in zip(
    axes.flat,
    [img, eroded, dilated, opened, closed, cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)],
    ["Original (noisy)", "Eroded", "Dilated", "Opened", "Closed", "Gradient"]
):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Finding Contours

In [ ]:
# Create an image with multiple shapes
shapes = np.zeros((300, 400), dtype=np.uint8)
cv2.rectangle(shapes, (20, 20), (100, 100), 255, -1)
cv2.circle(shapes, (200, 70), 50, 255, -1)
cv2.ellipse(shapes, (330, 70), (40, 25), 0, 0, 360, 255, -1)
pts = np.array([[150, 200], [200, 280], [100, 280]], dtype=np.int32)
cv2.fillPoly(shapes, [pts], 255)
cv2.rectangle(shapes, (250, 180), (380, 280), 255, -1)

contours, hierarchy = cv2.findContours(shapes, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)

output = cv2.cvtColor(shapes, cv2.COLOR_GRAY2BGR)
cv2.drawContours(output, contours, -1, (0, 255, 0), 2)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.title(f"Found {len(contours)} contours")
plt.axis("off")
plt.show()

## 3. Contour Properties

In [ ]:
output = cv2.cvtColor(shapes, cv2.COLOR_GRAY2BGR)

for i, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, closed=True)
    x, y, w, h = cv2.boundingRect(cnt)
    circularity = 4 * np.pi * area / (perimeter ** 2) if perimeter > 0 else 0
    
    # Draw bounding box and label
    cv2.rectangle(output, (x, y), (x + w, y + h), (0, 0, 255), 2)
    cv2.putText(output, f"#{i} A={area:.0f} C={circularity:.2f}",
                (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    
    print(f"Contour {i}: area={area:.1f}, perimeter={perimeter:.1f}, "
          f"circularity={circularity:.3f}, aspect_ratio={w/h:.2f}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.title("Contour Properties")
plt.axis("off")
plt.show()

## Exercises

1. Use morphological operations to clean a noisy binary image, then find contours
2. Classify shapes by circularity (circle vs. non-circle) using a threshold
3. Compute and draw the convex hull for each detected contour

In [ ]:
# Your code here
